# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ellarh101/ML-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q pandas numpy

In [2]:
import os, subprocess
import numpy as np
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("ML-starter"):
        subprocess.run(["git", "clone", "https://github.com/Ellarh101/ML-starter"], check=True)
    os.chdir("ML-starter")

os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
print((df["avg_position"] == 0).sum(), "rows with no position data")

(30000, 44)
1205 rows with no position data


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule, in plain words
Refresh a page that has sat untouched, is already visible in search, and ranks just below the top spots, close enough that a rewrite could win clicks.

### Score = visibility × staleness × upside
Each part is between 0 and 1, and the levels are hand-set (no fitted weights). Multiplying means a page must do well on all three.
- **visibility:** percentile rank of log(`impressions_90d`)
- **staleness:** percentile rank of `days_since_last_update`
- **upside** (room to climb): 1.0 at position 4–10, 0.8 at 11–20, 0.3 at 1–3 (little room left) and 21–50 (clicks collapse, see Signal 2), 0 with no position data

### Reason code and action (one per row, first match wins)
| Condition | Reason code | Action |
|---|---|---|
| stale ≥ 91 days, impressions ≥ 500, position 4–20 | `STALE_NEAR_PAGE_ONE` | `REFRESH` |
| stale ≥ 91 days, impressions ≥ 500 | `STALE_VISIBLE` | `REVIEW` |
| anything else | `LOW_PRIORITY` | `MONITOR` |

91 days is where the dictionary's `91-180` tier starts (only 133 pages are 181+). 500 impressions is the volume floor from Signal 2.

### Never used as inputs
`trend_pct`, `trend_direction`, `is_declining_label` and all `*_30d` columns. The label is built from them. They appear only after the fact, to judge the rule.

### Signal check 1: staleness (behind the refresh flags)
Do pages left un-updated for longer decline more often? The outcome is the share with `trend_direction == "down"`, used for checking only, never as a rule input. Only pages with impressions in the earlier 30 days are counted.

In [3]:
fresh_order = ["0-30", "31-90", "91-180", "181+"]
base = df[df["impressions_prev_30d"] > 0].copy()
base["declined"] = (base["trend_direction"] == "down").astype(int)

t1 = (base.groupby("freshness_tier")
        .agg(n=("content_id", "count"),
             pct_declining=("declined", "mean"),
             median_impr_90d=("impressions_90d", "median"),
             median_age_days=("content_age_days", "median"))
        .reindex(fresh_order))
t1["pct_declining"] = (t1["pct_declining"] * 100).round(1)

print("overall % declining:", round(base["declined"].mean() * 100, 1), "| n =", len(base))
display(t1)

overall % declining: 61.1 | n = 26612


,n,pct_declining,median_impr_90d,median_age_days
freshness_tier,,,,
0-30,17454,60.0,762.0,180.0
31-90,167,61.7,526.0,263.0
91-180,8858,63.3,1839.0,263.0
181+,133,61.7,26.0,301.0


**Verdict: MIXED**

- The decline rate rises from 60.0% (0–30 days, n = 17,454) to 63.3% (91–180 days, n = 8,858). The gap is small but real, because n is large.
- It does not keep rising: 181+ days is 61.7%, but n is only 133, too few to tell.
- Caveat: staler pages are also older and more visible (median age 180 → 263 days; median impressions 762 → 1,839), so staleness is tangled with age and traffic.
- **Effect on my rule:** staleness stays as a weak ingredient. The cutoff is 91 days because almost no pages reach 180.

### Signal check 2: CTR vs position (behind the CTR-fix logic)
Does click-through rate (CTR) fall as a page ranks worse? Only pages with at least 500 impressions are counted, because CTR is noisy on tiny pages. Pages with no position data are excluded.

In [4]:
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
s2 = df[df["position_tier"].isin(pos_order) & (df["impressions_90d"] >= 500)]
g = s2.groupby("position_tier")

t2 = pd.DataFrame({
    "n": g.size(),
    "median_ctr_pct": g["ctr"].median().round(2),
    "pooled_ctr_pct": (g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100).round(2),
    "median_impr": g["impressions_90d"].median(),
}).reindex(pos_order)

print("n =", len(s2))
display(t2)

n = 16726


,n,median_ctr_pct,pooled_ctr_pct,median_impr
position_tier,,,,
top_3,458,0.20,0.49,4263.0
page_1,7064,0.24,0.35,4504.0
striking,4485,0.17,0.35,2174.0
page_3_5,4330,0.09,0.16,2465.0
deep,389,0.00,0.04,965.0


**Verdict: CONFIRMED**

- Pooled CTR falls from 0.49% (top_3, n = 458) to 0.35% (page_1 and striking, n = 7,064 and 4,485), then 0.16% (page_3_5) and 0.04% (deep, n = 389). The biggest drop comes beyond position 20.
- Caveats: page_1 and striking tie at 0.35%, and the top_3 median (0.20%) is below page_1 (0.24%). All CTRs here are far below the 2.78% the dictionary quotes for warehouse scale, so I read direction only.
- **Effect on my rule:** position matters mainly as inside vs outside the top 20. The levels inside the top 20 are judgment calls.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
d = df.copy()

d["visibility"] = np.log1p(d["impressions_90d"]).rank(pct=True)
d["staleness"]  = d["days_since_last_update"].rank(pct=True)

# Room to climb: hand-set levels, no fitted weights
p = d["avg_position"]
d["upside"] = np.select(
    [p <= 0, p < 4, p <= 10, p <= 20, p <= 50],
    [0.0,    0.3,   1.0,     0.8,     0.3],
    default=0.0)

d["score"] = d["visibility"] * d["staleness"] * d["upside"]

stale   = d["days_since_last_update"] >= 91     # the dictionary's 91-180 tier boundary
visible = d["impressions_90d"] >= 500
near    = (p >= 4) & (p <= 20)

d["reason_code"] = np.select(
    [stale & visible & near, stale & visible],
    ["STALE_NEAR_PAGE_ONE", "STALE_VISIBLE"],
    default="LOW_PRIORITY")
d["action"] = d["reason_code"].map(
    {"STALE_NEAR_PAGE_ONE": "REFRESH", "STALE_VISIBLE": "REVIEW", "LOW_PRIORITY": "MONITOR"})

d["rank"] = d["score"].rank(method="first", ascending=False).astype(int)
print(d["reason_code"].value_counts())

reason_code
LOW_PRIORITY           23425
STALE_NEAR_PAGE_ONE     3835
STALE_VISIBLE           2740
Name: count, dtype: int64


In [6]:
cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
        "impressions_90d", "days_since_last_update", "avg_position", "ctr",
        "content_type", "main_intent"]
queue = d.sort_values("rank")[cols]
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")

wrote 30000 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
pd.set_option("display.width", 200)
top20 = queue.head(20)
display(top20)

,rank,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,content_type,main_intent
10870,1,content_a5dbb404bdc2,client_f369cb89fc,0.982477,STALE_NEAR_PAGE_ONE,REFRESH,79035,106,8.7,0.07,keyword article,informational
16648,2,content_69fad7e6c50c,client_7f2253d7e2,0.951654,STALE_NEAR_PAGE_ONE,REFRESH,28000,106,4.7,1.32,keyword article,informational
22197,3,content_6ac3ab740bbf,client_f369cb89fc,0.940174,STALE_NEAR_PAGE_ONE,REFRESH,22462,106,4.6,0.14,keyword article,informational
13070,4,content_c8f7e3b9891c,client_7f2253d7e2,0.890999,STALE_NEAR_PAGE_ONE,REFRESH,12029,106,5.8,1.01,keyword article,informational
14154,5,content_ee219ac0c645,client_7f2253d7e2,0.867328,STALE_NEAR_PAGE_ONE,REFRESH,9594,106,5.1,0.75,keyword article,informational
28669,6,content_df1e8cee858c,client_f369cb89fc,0.863017,STALE_NEAR_PAGE_ONE,REFRESH,9188,106,4.9,0.09,keyword article,commercial
29896,7,content_ae62a4fba956,client_7f2253d7e2,0.848349,STALE_NEAR_PAGE_ONE,REFRESH,8012,106,8.1,0.24,keyword article,informational
6653,8,content_5fe46e04994d,client_4e07408562,0.843200,STALE_NEAR_PAGE_ONE,REFRESH,517715,104,4.2,0.14,keyword article,informational
13537,9,content_2c2606c5d176,client_19581e27de,0.843003,STALE_NEAR_PAGE_ONE,REFRESH,347399,104,4.2,0.53,keyword article,commercial
26531,10,content_cb112fce36be,client_19581e27de,0.842919,STALE_NEAR_PAGE_ONE,REFRESH,309910,104,5.6,0.16,keyword article,transactional


All 20 rows: action `REFRESH`, reason `STALE_NEAR_PAGE_ONE`. Confidence is capped at medium because staleness (Signal 1) was only a weak signal.

| # | Why it's there | What would make it wrong | Conf. |
|---|---|---|---|
| 1 | 79,035 impressions, position 8.7, CTR 0.07% | The answer is shown in the results, so no extra clicks to win | med |
| 2 | 28,000 impressions, position 4.7, CTR 1.32% | CTR is already well above typical page one (~0.35%); a rewrite could hurt a page that works | low |
| 3 | 22,462 impressions, position 4.6, CTR 0.14% | Ads or answer boxes, not the page, cause the low CTR | med |
| 4 | 12,029 impressions, position 5.8 | The impressions come from a short spike, not a steady audience | med |
| 5 | 9,594 impressions, position 5.1 | 5.1 is an average (e.g. rank 1 for some queries, 30 for others) | med |
| 6 | 9,188 impressions, position 4.9, CTR 0.09%, commercial | Shopping results and ads take the clicks | med |
| 7 | 8,012 impressions, position 8.1 | 106 days is one client's shared batch date, not neglect (ranks 2, 4, 5, 7 match) | low |
| 8 | 517,715 impressions (largest), position 4.2, CTR 0.14% | Rewrite risk outweighs the gain; a person should review it first | med |
| 9 | 347,399 impressions, position 4.2, CTR 0.53%, commercial | CTR already above typical page one, so little headroom | med |
| 10 | 309,910 impressions, position 5.6, CTR 0.16%, transactional | Buyers click product listings, not articles | med |
| 11 | 295,097 impressions, position 7.3, CTR 0.05% | An AI overview or snippet answers it without a click | med |
| 12 | 211,366 impressions, position 5.1 | 104 days is this client's site-wide update date | low |
| 13 | 208,678 impressions, position 9.7, CTR 0.00% | Clicks are not being recorded, so a refresh fixes nothing | low |
| 14 | 201,584 impressions, position 5.8, commercial | The traffic is a seasonal spike | med |
| 15 | 201,111 impressions, position 5.7 | Evergreen content where age does not hurt | med |
| 16 | 190,623 impressions, position 4.3, commercial | The impressions come from queries unrelated to the target keyword | med |
| 17 | 187,893 impressions, position 4.0, transactional | Position sits exactly on my 4.0 cutoff; at 3.9 it would score far lower | low |
| 18 | 181,574 impressions, position 6.2 | It competes with another page from the same client; merging is the fix | med |
| 19 | 176,296 impressions, position 4.3, CTR 0.23%, commercial | 0.23% may be normal for this results page, so no gap to close | med |
| 20 | 174,408 impressions, position 4.7, CTR 0.56% | CTR already above typical page one, so small headroom | med |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
used = ["impressions_90d", "days_since_last_update", "avg_position"]
forbidden = ["trend_pct", "trend_direction", "is_declining_label"] + \
            [c for c in df.columns if c.endswith("_30d") or "prev_30d" in c]
print("leaky inputs used:", [c for c in used if c in forbidden])   # must print []

# After-the-fact only: the label is used here to JUDGE the rule, never to build it
d["declined"] = (d["trend_direction"] == "down").astype(int)
ranked = d.sort_values("rank")
base_rate = d["declined"].mean()
for k in (20, 50):
    print(f"top-{k} declining rate: {ranked.head(k)['declined'].mean():.3f} | base rate: {base_rate:.3f}")

# Who fills the top 20? (patterns for the weak-picks paragraph)
t20 = ranked.head(20)
print(t20["client_id"].value_counts().head(5))
print(t20["content_type"].value_counts())

print(d[used].isna().sum())   # should be all zeros

leaky inputs used: []
top-20 declining rate: 0.650 | base rate: 0.542
top-50 declining rate: 0.480 | base rate: 0.542
client_id
client_19581e27de    11
client_7f2253d7e2     4
client_f369cb89fc     3
client_4e07408562     1
client_6208ef0f77     1
Name: count, dtype: int64
content_type
keyword article    20
Name: count, dtype: int64
impressions_90d           0
days_since_last_update    0
avg_position              0
dtype: int64


### Weak picks
- **Rank 2:** CTR is already 1.32%, far above typical page one, so it needs a refresh least.
- **Rank 13:** 0.00% CTR on 208k impressions looks like a tracking gap, not a page problem.
- **Rank 17:** position is exactly 4.0, my upside cutoff, so its place is partly a boundary effect.

### Patterns
- 11 of the top 20 come from one client, and all 20 are keyword articles. A per-client cap would diversify the queue.
- Staleness barely separates pages (all 104–106 days), so ranks 8–20 are near-ties (scores within about 0.001).

### Does the queue pick declining pages?
Judged after the fact; the label is never an input.

| | Declined |
|---|---|
| Top 20 | 65.0% |
| Top 50 | 48.0% |
| All pages (base rate) | 54.2% |

With so few rows, both figures are within noise of the base rate. There is no evidence this rule finds declining pages: it ranks refresh upside, not decline risk. This is the baseline my Week-5 model must beat.

### Leakage check
No label or `*_30d` columns are used (the check prints `[]`). One caveat: `impressions_90d` overlaps the last 30 days, which define the label, so every result here is observed, directional, and for decision-support only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.